In [128]:
import math
import numpy as np
N = 1000
Z = np.random.poisson(2.9,N)
strata = range(0,7)

# Number of simulations in each stratum
sample_sizes = np.array(
    [np.sum(Z == i) for i in range(6)] +
    [np.sum(Z >= 6)]
)
# probabilities of each stratum for i = 0..5 and i >= 6
p = [np.exp(-2.9)*2.9**i / math.factorial(i) for i in strata[:6]]
p = np.array(p + [1 - sum(p)]) # P(Z = i)


Ybar = np.zeros(len(strata))
sigma2 = np.zeros(len(strata))
for i in strata:
    # Make N_i samples of
    if sample_sizes[i] == 0:
        continue

    if i == 0:
        # Empty sum
        X = np.zeros(sample_sizes[i])
    elif i < 6:
        # Find the conditional probability, given Z = i
        X = np.sum(np.random.exponential(1,size = (sample_sizes[i], i ))**2, axis = 1)

    else:
        # Consider, when Z >= 6, then we must draw at random for each os th
        Z_ge6 = Z[Z >= 6]

        X = np.array(
            [np.sum(np.random.exponential(1,size = z)**2) for z in Z_ge6]
        )

    # Compute condition
    Y = X < 1
    # Mean for each strata
    Ybar[i] = np.sum(Y) / sample_sizes[i]
    sigma2[i] = np.sum((Y - Ybar[i])**2) / (sample_sizes[i] - 1)
# Finally estimate by summing over the weighted means
ls = np.sum(p*Ybar)
sigma_str2 = np.sum(sigma2*p**2/sample_sizes)

print("Estimate: ", ls)
print("Variance: ", sigma_str2)

Estimate:  0.28982109776742
Variance:  0.00013244994853178104


We can now compare this with the crude method.

In [129]:
Z = np.random.poisson(2.9,N)
X = np.array([np.sum(np.random.exponential(1,size = z)**2) for z in Z])
Y = X < 1
ls_crude = np.mean(Y)
sigma_crude2 = np.var(Y) / N
print("Estimate: ", ls_crude)
print("Variance: ", sigma_crude2)
print("Improvement: ", sigma_crude2/sigma_str2)

Estimate:  0.281
Variance:  0.00020203900000000005
Improvement:  1.5253988562443366


If we want a further improvement, we can exploit the fact the P(X < 1 | Z = 0) = 1, and thus we can use this to reduce the variance of the estimator, as we can just use the closed form.

$$P(X < 1) = P(X < 1 | Z = 0)P(Z = 0) + P(X < 1 | Z = 1) + ...$$
Such that
$$ P(X < 1) = 1\cdot P(Z = 0  + P(X < 1 | Z = 1) + ...$$

In [138]:
Ni = np.round(N * p[1:] / (1 - p[0])).astype(int)
for i in strata[1:]:
    # Make N_i samples of each
    if i < 6:
        # Find the conditional probability, given Z = i
        X = np.sum(np.random.exponential(1,size = (Ni[i-1], i ))**2, axis = 1)

    else:
         # Generate ni values from Z | Z >= 6
        Z_ge6 = []

        while len(Z_ge6) < Ni[i-1]:
            z = np.random.poisson(2.9)
            if z >= 6:
                Z_ge6.append(z)

        X = np.array([
            np.sum(np.random.exponential(1, size=z)**2)
            for z in Z_ge6
        ])

    # Compute condition
    Y = X < 1
    # Mean for each strata
    Ybar[i] = np.sum(Y) / Ni[i-1]
    sigma2[i] = np.sum((Y - Ybar[i])**2) / (Ni[i-1] - 1)
# Estimate using closed form for P(X < 1 | Z = 0)*P(Z = 0) = 1*e^-2.9
ls = np.e**-2.9 + np.sum(p[1:]*Ybar[1:])
sigma_str2 = np.sum(sigma2[1:]*p[1:]**2/Ni)

print("Estimate: ", ls)
print("Variance: ", sigma_str2)

Estimate:  0.29104406263326094
Variance:  0.00013197489450768115
